# 10C1 — Read saved sccomp significance and diagnostics (no refitting)

The real 10C run already calls `sccomp_test()` and saves one `contrasts.csv` per
composition model. The observed-fraction plots are descriptive; existing contrast
forest plots use a color for `c_FDR < 0.05` but do not print each value. This notebook
loads the saved tables, displays the exact numbers and optionally produces labelled
forest plots. **No R, sccomp package, Stan/GPU task, count matrix or model rerun is needed.**

Set `SCCOMP_ROOT` to the real 10C fit directory containing `sccomp_fit_summary.csv`.
Do not use 10C0's synthetic smoke-test directory. Different fits/thresholds have
separate roots; this notebook never selects the latest one implicitly.

## Interpretation
- `c_effect`: native-scale composition contrast estimate (posterior median in the
  sccomp 2.2.0 implementation), not DESeq2 log2FC or percentage-point difference.
- `c_lower`, `c_upper`: the saved posterior credible interval (95% with the default
  `percent_false_positive=5` used by this runner).
- `c_pH0`: posterior error probability against a directional effect beyond the
  effect threshold; not a frequentist P value or simply the central-interval mass.
- `c_FDR`: sccomp's Bayesian FDR, used for screening composition changes.
- `c_` denotes composition. `v_` concerns variability, which was modeled with an
  intercept only; it is not the primary treatment-mean composition result.

The configured effect threshold defaults to 0.1 in the original 10C, in the native
coefficient scale. It is not ten percentage points. The notebook reads the actual
saved value, and does not retest at a different effect threshold.

In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display

# Paste the SCCOMP_ROOT printed by the REAL 10C fit cell, not the 10C0 synthetic folder.
# In the still-open 10C kernel, the existing SCCOMP_ROOT can be reused directly.
SCCOMP_ROOT = globals().get('SCCOMP_ROOT', None)
# SCCOMP_ROOT = Path('/your/characterization/output/sccomp/<fit-fingerprint>')

FDR_CUTOFF = 0.05
if not 0 < FDR_CUTOFF < 1:
    raise ValueError('FDR_CUTOFF must be between 0 and 1.')
if SCCOMP_ROOT is None:
    raise ValueError('Set SCCOMP_ROOT to the folder printed by notebook 10C (containing sccomp_fit_summary.csv).')
SCCOMP_ROOT = Path(SCCOMP_ROOT).expanduser().resolve()
manifest_path = SCCOMP_ROOT / 'sccomp_fit_summary.csv'
if not manifest_path.is_file():
    raise FileNotFoundError(f'{manifest_path}: select the REAL 10C model root, not the smoke test or images folder.')
manifest = pd.read_csv(manifest_path, keep_default_na=False)
if not {'model','status'}.issubset(manifest.columns) or manifest['model'].duplicated().any():
    raise ValueError('Unexpected or ambiguous sccomp model manifest.')
model_spec_path = SCCOMP_ROOT/'model_spec.json'
model_spec = json.loads(model_spec_path.read_text()) if model_spec_path.is_file() else {}
parts, model_reports, source_files = [], [], [manifest_path]
if model_spec_path.is_file(): source_files.append(model_spec_path)
for rec in manifest.to_dict('records'):
    name = str(rec['model'])
    if Path(name).name != name:
        raise ValueError('Invalid model directory name.')
    folder = SCCOMP_ROOT/name
    table_path = folder/'contrasts.csv'
    meta_path = folder/'complete_R.json'
    info = json.loads(meta_path.read_text()) if meta_path.is_file() else {}
    status = str(info.get('status', 'completion_manifest_missing'))
    if meta_path.is_file(): source_files.append(meta_path)
    if info.get('model', name) != name:
        raise ValueError(f'{name}: completion manifest identifies another model.')
    if rec.get('job_id') and info.get('job_id') and rec['job_id'] != info['job_id']:
        raise ValueError(f'{name}: job fingerprint differs between fit summary and completion manifest.')
    record = dict(model=name, fit_summary_status=rec['status'], diagnostic_status=status,
                  effect_threshold=info.get('effect_threshold', model_spec.get('settings',{}).get('effect_threshold',np.nan)),
                  threshold_argument=info.get('threshold_argument','not recorded'),
                  max_rhat=info.get('max_rhat',np.nan), n_divergences=info.get('n_divergences',np.nan),
                  n_mice=info.get('n_mice',np.nan), table=str(table_path))
    if not table_path.is_file():
        record.update(n_tests=0,n_FDR_pass=0,n_FDR_and_basic_gate=0,n_FDR_pass_needing_review=0)
        model_reports.append(record)
        print('No contrast table:', name, '| fit status:', rec['status'])
        continue
    x = pd.read_csv(table_path)
    required = {'cell_group','parameter','c_effect','c_lower','c_upper','c_pH0','c_FDR'}
    if not required.issubset(x.columns):
        raise KeyError(f'{name}: missing {sorted(required-set(x.columns))}; no guessed p-value column used.')
    if x.duplicated(['cell_group','parameter']).any():
        raise ValueError(f'{name}: duplicate category/contrast rows.')
    for c in ['c_effect','c_lower','c_upper','c_pH0','c_FDR']:
        x[c] = pd.to_numeric(x[c], errors='raise')
    for c in ['c_pH0','c_FDR']:
        v=x[c].dropna()
        if not np.isfinite(v).all() or ((v<0)|(v>1)).any():
            raise ValueError(f'{name}: invalid probability values in {c}.')
    source_files.append(table_path)
    x.insert(0,'model',name)
    audit_path=folder/'contrast_expression_audit.csv'
    if audit_path.is_file():
        audit=pd.read_csv(audit_path)
        if not {'contrast','numerator','denominator'}.issubset(audit) or audit.contrast.duplicated().any():
            raise ValueError('Unexpected contrast expression audit.')
        x=x.merge(audit[['contrast','numerator','denominator']],left_on='parameter',right_on='contrast',how='left',validate='many_to_one')
        if x['numerator'].isna().any() or x['denominator'].isna().any():
            raise ValueError(f'{name}: contrast labels do not match the saved expression audit.')
        source_files.append(audit_path)
    x['diagnostic_status']=status
    x['fit_summary_status']=rec['status']
    x['effect_threshold']=record['effect_threshold']
    x['max_rhat']=record['max_rhat']; x['n_divergences']=record['n_divergences']
    x['passes_c_FDR']=x['c_FDR'].notna() & (x['c_FDR'] < FDR_CUTOFF)
    interval_ok=np.isfinite(x[['c_effect','c_lower','c_upper']]).all(axis=1) & (x.c_lower<=x.c_effect) & (x.c_effect<=x.c_upper)
    basic_ok=(status=='diagnostics_pass_basic' and rec['status']=='diagnostics_pass_basic')
    x['basic_diagnostics_pass']=basic_ok
    x['finite_consistent_interval']=interval_ok
    x['FDR_and_basic_gate']=x['passes_c_FDR'] & x['basic_diagnostics_pass'] & interval_ok
    x['direction_of_estimate']=np.select([x.c_effect>0,x.c_effect<0],['positive numerator-minus-denominator','negative numerator-minus-denominator'],default='zero or unavailable')
    x['assessment']=np.select(
        [~x['c_FDR'].notna(), ~interval_ok, x['passes_c_FDR'] & ~x['basic_diagnostics_pass'], x['FDR_and_basic_gate']],
        ['FDR unavailable','Review invalid/undefined effect interval','FDR passes, but review convergence/completion','FDR passes and basic diagnostic gate passes'],
        default='Does not pass selected FDR cutoff')
    parts.append(x)
    record.update(n_tests=len(x),n_FDR_pass=int(x.passes_c_FDR.sum()),
                  n_FDR_and_basic_gate=int(x.FDR_and_basic_gate.sum()),
                  n_FDR_pass_needing_review=int((x.passes_c_FDR & ~x.FDR_and_basic_gate).sum()))
    model_reports.append(record)
if not parts:
    raise RuntimeError('No model has a usable contrasts.csv. Check sccomp_fit_summary.csv and each R_run.log.')
all_sccomp = pd.concat(parts,ignore_index=True).sort_values(['model','parameter','c_FDR','cell_group'],na_position='last')
model_status = pd.DataFrame(model_reports)
print('Model diagnostics and counts:')
display(model_status.drop(columns='table'))
print('Original estimates and Bayesian FDR values:')
view_cols=[c for c in ['model','cell_group','parameter','c_effect','c_lower','c_upper','c_pH0','c_FDR','effect_threshold','assessment'] if c in all_sccomp]
with pd.option_context('display.max_rows',100,'display.float_format',lambda v:f'{v:.4g}'):
    display(all_sccomp[view_cols])

## Convergence comes before a biological significance claim
`diagnostics_pass_basic` is the existing runner's basic screen: finite maximum
R-hat <=1.05 and no recorded divergences. It is NOT a guarantee that every diagnostic
is satisfactory. Inspect `Stan_parameter_summary.csv` for ESS/R-hat,
`Stan_diagnostics.txt`, traces and posterior predictive checks. Contrast-row R-hat
can be missing because contrast labels are not base parameter names; use fitted
parameter summaries rather than treating missing values as a pass.

A low `c_FDR` remains visible even when diagnostics need review. The table separates
FDR-passing rows from rows that also pass the existing basic gate, without silently
suppressing model problems. A failed/not-testable model is not 'nonsignificant'.

In [ ]:
# Show FDR-passing rows, WITHOUT hiding convergence problems.
passed_FDR = all_sccomp.loc[all_sccomp['passes_c_FDR']].copy()
passed_basic_gate = all_sccomp.loc[all_sccomp['FDR_and_basic_gate']].copy()
needs_review = passed_FDR.loc[~passed_FDR['FDR_and_basic_gate']].copy()
print(f'{len(passed_FDR)} rows have c_FDR < {FDR_CUTOFF}.')
print(f'{len(passed_basic_gate)} also pass the runner\'s BASIC diagnostic gate.')
print(f'{len(needs_review)} FDR-passing rows need convergence/completion/interval review.')
display(passed_FDR[view_cols])

# Optional focused view. Empty list means all six comparisons.
MODEL_TO_VIEW = 'overall_tumor_fraction'
CELL_GROUPS_TO_VIEW = ['All tumor cell']
CONTRASTS_TO_VIEW = []
# CONTRASTS_TO_VIEW = ['conditionAdagrasib - conditionVehicle',
#                      'conditionCombination - conditionAntiPD1']
focused=all_sccomp.loc[all_sccomp['model'].eq(MODEL_TO_VIEW)]
if CELL_GROUPS_TO_VIEW:
    focused=focused.loc[focused['cell_group'].isin(CELL_GROUPS_TO_VIEW)]
if CONTRASTS_TO_VIEW:
    focused=focused.loc[focused['parameter'].isin(CONTRASTS_TO_VIEW)]
display(focused[view_cols])

In [ ]:
# Export a NEW report folder. Original contrasts, model fits and completion files stay unchanged.
def content_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
review_spec=dict(fdr_cutoff=FDR_CUTOFF,source_files={str(p):content_hash(p) for p in source_files},
                basic_gate='Existing runner status only; not independent certification of fit adequacy',
                multiplicity='Original sccomp c_FDR retained; no new adjustment or threshold retesting')
review_id=hashlib.sha256(json.dumps(review_spec,sort_keys=True).encode()).hexdigest()[:12]
REVIEW_OUT=SCCOMP_ROOT/'statistical_results_review'/review_id
REVIEW_OUT.mkdir(parents=True,exist_ok=True)
(REVIEW_OUT/'review_spec.json').write_text(json.dumps(review_spec,indent=2))
all_sccomp.to_csv(REVIEW_OUT/'all_sccomp_contrasts.csv',index=False)
model_status.to_csv(REVIEW_OUT/'model_diagnostic_review.csv',index=False)
passed_FDR.to_csv(REVIEW_OUT/'cFDR_pass_all_diagnostic_statuses.csv',index=False)
passed_basic_gate.to_csv(REVIEW_OUT/'cFDR_pass_and_basic_diagnostics_pass.csv',index=False)
needs_review.to_csv(REVIEW_OUT/'cFDR_pass_but_review_required.csv',index=False)
print('Saved reports:',REVIEW_OUT)

In [ ]:
# Optional forest plots with exact numerical c_FDR labels, no model refitting.
import matplotlib.pyplot as plt
import re
MAKE_LABELLED_FOREST_PLOTS=True
if MAKE_LABELLED_FOREST_PLOTS:
    plot_dir=REVIEW_OUT/'labelled_forest_plots';plot_dir.mkdir(exist_ok=True)
    for (model,contrast),group in all_sccomp.groupby(['model','parameter'],sort=False):
        group=group.sort_values('c_effect',na_position='last').reset_index(drop=True)
        fig,ax=plt.subplots(figsize=(11,max(4,.45*len(group)+2)))
        finite=group['finite_consistent_interval']
        for i,row in group.iterrows():
            if row.finite_consistent_interval:
                ax.plot([row.c_lower,row.c_upper],[i,i],linewidth=1.2)
                # Filled circle: original FDR passes. Open circle: not passing/NA.
                ax.plot(row.c_effect,i,marker='o',markersize=7,linestyle='none',
                        fillstyle='full' if row.passes_c_FDR else 'none')
            label='NA' if pd.isna(row.c_FDR) else f'{row.c_FDR:.3g}'
            label+=' *' if row.passes_c_FDR else ''
            if row.passes_c_FDR and not row.FDR_and_basic_gate: label+=' [REVIEW]'
            ax.text(1.02,i,label,transform=ax.get_yaxis_transform(),va='center',fontsize=9)
        ax.axvline(0,linestyle='-',linewidth=.7)
        thresholds=pd.to_numeric(group.effect_threshold,errors='coerce').dropna().unique()
        if len(thresholds)==1 and np.isfinite(thresholds[0]):
            for edge in [-float(thresholds[0]),float(thresholds[0])]: ax.axvline(edge,linestyle='--',linewidth=.7)
        ax.set_yticks(range(len(group)));ax.set_yticklabels(group.cell_group)
        ax.set_ylim(-.7,len(group)-.3);ax.invert_yaxis()
        ax.set_xlabel('sccomp composition contrast (native model coefficient scale)')
        ax.set_title(f'{model}\n{contrast}\n{group.diagnostic_status.iloc[0]}')
        ax.text(1.02,1.02,'c_FDR',transform=ax.transAxes,fontweight='bold',fontsize=9)
        fig.text(.02,.02,f'* c_FDR < {FDR_CUTOFF}; intervals are Bayesian 95% credible intervals.\n'
                 '[REVIEW]: FDR passed but diagnostic/completion/interval review remains. No statistics recomputed.',fontsize=8)
        fig.subplots_adjust(left=.24,right=.79,bottom=.22,top=.78)
        safe=re.sub(r'[^A-Za-z0-9_.-]+','_',model+'__'+contrast).strip('_')
        fig.savefig(plot_dir/(safe+'.png'),dpi=300,bbox_inches='tight');plt.close(fig)
    print('Labelled contrast plots:',plot_dir)

## Three denominators, three different questions

| Model | Meaning |
|---|---|
| `overall_tumor_fraction` | All tumor cells versus other reviewed cells; tumor fraction among all reviewed cells |
| `subtypes_global_fraction` | Tumor 1–5 and other reviewed cells; each subtype among all reviewed cells |
| `subtypes_within_tumor` | Tumor 1–5 only; each subtype conditional on tumor cells |

For `conditionAdagrasib - conditionVehicle`, positive values mean a higher native
relative-composition coefficient for Adagrasib; negative values mean lower. Look at
the observed fractions/posterior predictions for a percentage-scale description.
A subtype can increase within remaining tumor while decreasing globally. The
complementary parts of the two-category model are not independent confirmations.

The sccomp 2.2 implementation computes c_FDR across cell groups within each contrast.
The saved FDR is not a joint correction over every contrast AND all three denominator
models. Do not concatenate Bayesian c_FDR values and feed them to a frequentist BH
routine as if they were raw P values. No multiplicity adjustment is altered here.

No result tables from the user's real fit were available while generating this
review notebook. CPU validation used synthetic saved-output fixtures only.

### Primary references
- Version-matched test implementation: https://raw.githubusercontent.com/bioconductor-source/sccomp/RELEASE_3_22/R/sccomp_test.R
- Stan diagnostics: https://mc-stan.org/learn-stan/diagnostics-warnings.html